In [0]:
import logging

In [0]:
logger = logging.getLogger("DatabricksWorkflow")
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
if not logger.hasHandlers():
    logger.addHandler(handler)

In [0]:
config_path = "dbfs:/configs/config.json"
try:
    config = spark.read.option("multiline", "true").json(config_path)
    logger.info(f"Successfully read config file from {config_path}")
except Exception as e:
    logger.error(f"Could not read config file at {config_path}: {e}", exc_info=True)
    raise FileNotFoundError(f"Could not read config file at {config_path}: {e}")

try:
    first_row = config.first()
    env = first_row["env"].strip().lower()
    lz_key = first_row["lz_key"].strip().lower()
    logger.info(f"Extracted configs: env={env}, lz_key={lz_key}")
except Exception as e:
    logger.error(f"Missing expected keys 'env' or 'lz_key' in config file: {e}", exc_info=True)
    raise KeyError(f"Missing expected keys 'env' or 'lz_key' in config file: {e}")

try:
    keyvault_name = f"ingest{lz_key}-meta002-{env}"
    logger.info(f"Constructed keyvault name: {keyvault_name}")
except Exception as e:
    logger.error(f"Error constructing keyvault name: {e}", exc_info=True)
    raise ValueError(f"Error constructing keyvault name: {e}")


In [0]:

try:
    client_secret = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-SECRET')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-CLIENT-SECRET from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-SECRET' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-SECRET' from Key Vault '{keyvault_name}': {e}")

try:
    tenant_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-TENANT-ID')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-TENANT-ID from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-TENANT-ID' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-TENANT-ID' from Key Vault '{keyvault_name}': {e}")

try:
    client_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-ID')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-CLIENT-ID from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-ID' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-ID' from Key Vault '{keyvault_name}': {e}")

logger.info("✅ Successfully retrieved all Service Principal secrets from Key Vault")


In [0]:
# --- Parameterise containers ---
curated_storage_account = f"ingest{lz_key}curated{env}"
curated_container = "gold"
silver_curated_container = "silver"

# --- Assign OAuth to storage accounts ---
storage_accounts = [curated_storage_account]

for storage_account in storage_accounts:
    try:
        configs = {
            f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
            f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
                "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
            f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
            f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
            f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
                f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
        }

        for key, val in configs.items():
            try:
                spark.conf.set(key, val)
            except Exception as e:
                logger.error(f"Failed to set Spark config '{key}' for storage account '{storage_account}': {e}", exc_info=True)
                raise RuntimeError(f"Failed to set Spark config '{key}' for storage account '{storage_account}': {e}")

        logger.info(f"✅ Successfully configured OAuth for storage account: {storage_account}")

    except Exception as e:
        logger.error(f"Error configuring OAuth for storage account '{storage_account}': {e}", exc_info=True)
        raise RuntimeError(f"Error configuring OAuth for storage account '{storage_account}': {e}")


In [0]:
active_case_note_document_segmentation = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/APPEALS/CDAM/stg_apl_combined")
active_case_note_document_segmentation.createOrReplaceTempView("active_case_note_document_segmentation")
 
active_case_note_documents_generated = spark.read.format("delta").load(f"abfss://{curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/APPEALS/CDAM/gold_appeals_with_html")
active_case_note_documents_generated.createOrReplaceTempView("active_case_note_documents_generated")

cdam_publish_payload_result = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/CDAM/publish_payload_audit")
cdam_publish_payload_result.createOrReplaceTempView("cdam_publish_payload_result")

cdam_call_result = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/CDAM/ack_audit")
cdam_call_result.createOrReplaceTempView("cdam_call_result")

In [0]:
ccd_publish_payload_result = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/publish_payload_audit")
ccd_publish_payload_result.createOrReplaceTempView("ccd_publish_payload_result")

ccd_call_result = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/all_active_states/ack_audit")
ccd_call_result.createOrReplaceTempView("ccd_call_result")

segmentation_state = spark.read.table("hive_metastore.gold_payloads.dashboard_segmentation_state")
segmentation_state.createOrReplaceTempView("segmentation_state")

valid_jsons = spark.read.table("hive_metastore.gold_payloads.dashboard_valid_json_files")
valid_jsons.createOrReplaceTempView("valid_jsons")

invalid_jsons = spark.read.table("hive_metastore.gold_payloads.dashboard_invalid_json_files")
invalid_jsons.createOrReplaceTempView("invalid_jsons")

In [0]:
case_link_groups = spark.read.table(f"hive_metastore.ariadm_active_appeals.silver_case_link_groups")
case_link_groups.createOrReplaceTempView("case_link_groups")
case_link_mappings = spark.read.table(f"hive_metastore.ariadm_active_appeals.aria_ccd_case_mappings")
case_link_mappings.createOrReplaceTempView("case_link_mappings")
case_link_combinations = spark.read.table(f"hive_metastore.ariadm_active_appeals.case_link_combinations")
case_link_combinations.createOrReplaceTempView("case_link_combinations")
case_link_payloads = spark.read.table(f"hive_metastore.ariadm_active_appeals.case_link_payloads")
case_link_payloads.createOrReplaceTempView("case_link_payloads")

ccd_case_link_publish_payload_result = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/CASE_LINKING/publish_payload_audit")
ccd_case_link_publish_payload_result.createOrReplaceTempView("ccd_case_link_publish_payload_result")

ccd_case_link_call_result = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/CASE_LINKING/ack_audit")
ccd_case_link_call_result.createOrReplaceTempView("ccd_case_link_call_result")

In [0]:
spark.sql("""
    WITH segmentation_state_cte AS (
        SELECT DISTINCT
            REPLACE(REPLACE(split_part(CaseNo, '.', 1), 'APPEALS_', ''), '_', '/') AS CaseNo,
            ariaDesiredState AS State,
            'Yes' AS `Segmentation State`
            -- RunID
        FROM segmentation_state
    ),

    valid_invalid_json_cte AS (
        SELECT
            CaseNo,
            State AS State,
            'Invalid' AS `Validation Status`
            -- RunID
        FROM valid_jsons

        UNION ALL

        SELECT
            CaseNo,
            State AS State,
            'Valid' AS `Validation Status`
            -- RunID
        FROM invalid_jsons
    )

    SELECT
        s.CaseNo AS CaseNo,
        a.State AS State,
        CASE WHEN s.CaseNo IS NOT NULL
            THEN 'Yes'
        END AS `Case Note Document Segmentation Status`,
        CASE WHEN g.CaseNo IS NOT NULL
            THEN 'SUCCESS'
        END AS `Case Note Document Generation Status`,
        p.Status AS `Case Note Document URL Publish Status`,
        r.Status AS `CDAM Call Status`,
        CASE WHEN a.State IS NOT NULL
            THEN 'Yes'
        END AS `State Segmentation Status`,
        b.`Validation Status`,
        c.Status AS `CCD Publish Payload Status`,
        d.Status AS `CCD Call Status`,
        p.FileName AS `Case Note Document File Name`,
        p.FileUrl AS `Case Note Document File URL`,
        p.PublishingDateTime AS `Case Note Document Publish Publishing Date Time`,
        p.Error AS `Case Note Document Publish Error`,
        r.StartDateTime AS `CDAM Call Function App Start Date Time`,
        r.EndDateTime AS `CDAM Call Function App End Date Time`,
        r.Error AS `CDAM Call Function App Error`,
        r.document_filename AS `CDAM document_filename`,
        r.document_url AS `CDAM document_url`,
        r.createdOn AS `CDAM createdOn`,
        c.PublishingDateTime AS `CCD Publish Payload Publishing Date Time`,
        c.Error AS `CCD Publish Payload Error`,
        d.StartDateTime AS `CCD Call Function App Start Date Time`,
        d.EndDateTime AS `CCD Call Function App End Date Time`,
        d.CCDCaseID AS `CCD Case Reference Number`,
        d.Error AS `CCD Call Function App Error`
    FROM active_case_note_document_segmentation s
        FULL OUTER JOIN active_case_note_documents_generated g ON s.CaseNo = g.CaseNo
        FULL OUTER JOIN cdam_publish_payload_result p ON s.CaseNo = p.CaseNo
        FULL OUTER JOIN cdam_call_result r ON s.CaseNo = r.CaseNo
        FULL OUTER JOIN segmentation_state_cte a ON s.CaseNo = a.CaseNo
        FULL OUTER JOIN valid_invalid_json_cte b ON s.CaseNo = b.CaseNo
        FULL OUTER JOIN ccd_publish_payload_result c ON s.CaseNo = c.AriaCaseNo
        FULL OUTER JOIN ccd_call_result d ON s.CaseNo = d.AriaCaseNo
    ORDER BY `CCD Publish Payload Publishing Date Time` DESC
""").display()


Databricks visualization. Run in Databricks to view.

In [0]:
spark.sql("""
     WITH segmentation_state_cte AS (
        SELECT DISTINCT
            REPLACE(REPLACE(split_part(CaseNo, '.', 1), 'APPEALS_', ''), '_', '/') AS CaseNo,
            ariaDesiredState AS State,
            'Yes' AS `Segmentation State`
            -- RunID
        FROM segmentation_state
    ),

    valid_invalid_json_cte AS (
        SELECT
            CaseNo,
            State AS State,
            'Invalid' AS `Validation Status`
            -- RunID
        FROM valid_jsons

        UNION ALL

        SELECT
            CaseNo,
            State AS State,
            'Valid' AS `Validation Status`
            -- RunID
        FROM invalid_jsons
    )

    SELECT 
        s.CaseNo AS CaseNo,
        a.State AS State,
        CASE WHEN s.CaseNo IS NOT NULL
            THEN 'Yes'
        END AS `Case Note Document Segmentation Status`,
        CASE WHEN g.CaseNo IS NOT NULL
            THEN 'SUCCESS'
        END AS `Case Note Document Generation Status`,
        p.Status AS `Case Note Document URL Publish Status`,
        r.Status AS `CDAM Call Status`,
        CASE WHEN a.State IS NOT NULL
            THEN 'Yes'
        END AS `State Segmentation Status`,
        b.`Validation Status`,
        c.Status AS `CCD Publish Payload Status`,
        d.Status AS `CCD Call Status`,
        CASE WHEN clg.CaseNo IS NOT NULL AND clg.LinkNo IS NOT NULL
            THEN 'YES'
            ELSE 'NO'
        END AS `ARIA Case Link Groups Exist`,
        CASE WHEN clm.CaseNo IS NOT NULL AND clm.CCDCaseID IS NOT NULL AND clm.CCDCaseID = t1.CCDCaseReferenceNumber
            THEN 'SUCCESS'
            ELSE 'ERROR'
        END AS `ARIA-CCD Case Mappings Status`,
        CASE WHEN clc.CCDCaseReferenceNumberFrom IS NOT NULL AND clc.CCDCaseReferenceNumberFrom = t1.CCDCaseReferenceNumber
            THEN 'SUCCESS'
            ELSE 'ERROR'
        END AS `Case Link Combinations Status`,
        CASE WHEN clp.CCDCaseReferenceNumberFrom IS NOT NULL AND clp.CCDCaseReferenceNumberFrom != '' THEN 'SUCCESS' ELSE 'ERROR' END AS `Case Link Payloads Status`,
        t1.Status AS `Case Link Publish Status`,
        t2.Status AS `Case Link CCD Call Status`,
        p.FileName AS `Case Note Document File Name`,
        p.FileUrl AS `Case Note Document File URL`,
        p.PublishingDateTime AS `Case Note Document Publish Publishing Date Time`,
        p.Error AS `Case Note Document Publish Error`,
        r.StartDateTime AS `CDAM Call Function App Start Date Time`,
        r.EndDateTime AS `CDAM Call Function App End Date Time`,
        r.Error AS `CDAM Call Function App Error`,
        r.document_filename AS `CDAM document_filename`,
        r.document_url AS `CDAM document_url`,
        r.createdOn AS `CDAM createdOn`,
        c.PublishingDateTime AS `CCD Publish Payload Publishing Date Time`,
        c.Error AS `CCD Publish Payload Error`,
        d.StartDateTime AS `CCD Call Function App Start Date Time`,
        d.EndDateTime AS `CCD Call Function App End Date Time`,
        d.CCDCaseID AS `CCD Case Reference Number`,
        d.Error AS `CCD Call Function App Error`,
        CASE WHEN clg.LinkNo IS NOT NULL
            THEN COUNT(*) OVER (PARTITION BY clg.LinkNo) - 1
            ELSE 0
        END AS `ARIA Case Link Groups Case Link Count`,
        t1.CaseLinkCount as `Case Link Publish Case Link Count`,
        t1.PublishingDateTime as `Case Link Publish Publishing Date Time`,
        t1.Error as `Case Link Publish Error`,
        t2.CaseLinkCount as `Case Link CCD Call Function App Case Link Count`,
        t2.StartDateTime as `Case Link CCD Call Function App Start Date Time`,
        t2.EndDateTime as `Case Link CCD Call Function App End Date Time`,
        t2.Error as `Case Link CCD Call Function App Error`
    FROM active_case_note_document_segmentation s
        FULL OUTER JOIN active_case_note_documents_generated g ON s.CaseNo = g.CaseNo
        FULL OUTER JOIN cdam_publish_payload_result p ON s.CaseNo = p.CaseNo
        FULL OUTER JOIN cdam_call_result r ON s.CaseNo = r.CaseNo
        FULL OUTER JOIN segmentation_state_cte a ON s.CaseNo = a.CaseNo
        FULL OUTER JOIN valid_invalid_json_cte b ON s.CaseNo = b.CaseNo
        FULL OUTER JOIN ccd_publish_payload_result c ON s.CaseNo = c.AriaCaseNo
        FULL OUTER JOIN ccd_call_result d ON s.CaseNo = d.AriaCaseNo
        FULL OUTER JOIN case_link_groups clg ON s.CaseNo = clg.CaseNo
        FULL OUTER JOIN case_link_mappings clm ON s.CaseNo = clm.CaseNo
        FULL OUTER JOIN (
            SELECT
                CaseNoFrom,
                MAX(CCDCaseReferenceNumberFrom) AS CCDCaseReferenceNumberFrom
            FROM case_link_combinations
            GROUP BY CaseNoFrom
        ) clc ON s.CaseNo = clc.CaseNoFrom
        FULL OUTER JOIN case_link_payloads clp ON clc.CCDCaseReferenceNumberFrom = clp.CCDCaseReferenceNumberFrom
        FULL OUTER JOIN ccd_case_link_publish_payload_result t1 ON clp.CCDCaseReferenceNumberFrom = t1.CCDCaseReferenceNumber
        FULL OUTER JOIN ccd_case_link_call_result t2 ON t1.CCDCaseReferenceNumber = t2.CCDCaseReferenceNumber AND t1.RunID = t2.RunID
    ORDER BY `CCD Publish Payload Publishing Date Time` DESC
""").display()

In [0]:
dbutils.notebook.exit("Notebook completed successfully")